# Week 2 · Tutorial 1 — The EDA checklist, variable types and missing values

**Covers videos 2.1 – 2.2:** what we are looking for · variable types and missing values (including values in disguise).

**Dataset:** `building_hourly.csv` — hourly electricity use of a commercial office building (`B-07`) in 2023. **One row is one hour.**

| column | meaning |
|---|---|
| `timestamp` | hour (UTC) |
| `electricity_kwh` | electricity use of the building in the hour |
| `temperature_c` | outdoor temperature |
| `occupancy` | people counted in the building |
| `co2_ppm` | indoor CO₂ concentration |
| `wing_code` | wing of the building (1–4) |
| `building_id` | building identifier |

**The checklist:** structure → types → missing → distributions → (pairs → relation to the target).
This tutorial covers the first three, plus the danger of *looking at too much*.

### How to work with this notebook (no programming experience needed)

* A notebook is a list of **cells**. Click a code cell and press **Shift + Enter** to run it; the result appears below it. Run the cells **from top to bottom**.
* Cells that start with `import` or read a file (`pd.read_csv(...)`) just load tools and data. You do not need to understand every line.
* In every **✏️ Your turn** block you make one small, exactly described change (a number, a word, or a line to paste in), then run the cell again.
  The box **What to change** shows the exact text to type. Lines marked `# TODO` are the ones to edit.
* Write down the numbers that appear — the quiz asks for them. If you break something, re-run the cells above it, or use *Kernel → Restart & Run All*.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = "https://raw.githubusercontent.com/blasern/data-tutorials/main/data/"   # where the data files live
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

### Explore the training set

Choosing what to look at is already a modelling decision. So we first split the year **by time** — the first 70 % of the hours
form the training set — and do all exploration on that part.

In [ ]:
raw = pd.read_csv(DATA_URL + "building_hourly.csv", parse_dates=["timestamp"])
raw = raw.sort_values("timestamp").reset_index(drop=True)
n_train = int(0.7 * len(raw))

train = raw.iloc[:n_train].copy()
print("training hours:", len(train), "| from", train["timestamp"].min(), "to", train["timestamp"].max())
train.head()

## 1. Structure and types: how many distinct values?

The number of distinct values (`nunique`) quickly tells us whether a numeric-looking column is really a **category**
(few values), a **constant** (one value, no information), or truly continuous.

In [ ]:
print(train.dtypes)
print()
train.nunique().sort_values()

### ✏️ Your turn 1

Classify the columns by their number of distinct values (training set only).

Fill in the `TODO`s:
- `constant_cols`: columns that have exactly **1** distinct value;
- `categorical_like`: columns that have **at most 10** distinct values *and are not constant* — they should be treated as categories even if stored as numbers;
- then convert the numeric-looking categorical column(s) to the `category` dtype.

**What to change** — edit the cell exactly like this, then run it:

Replace
```python
constant_cols = []
categorical_like = []
```
by
```python
constant_cols = list(n_unique[n_unique == 1].index)
categorical_like = list(n_unique[(n_unique > 1) & (n_unique <= 10)].index)
```

In [ ]:
n_unique = train.nunique()

constant_cols = []        # TODO
categorical_like = []     # TODO

print("constant columns   :", constant_cols)
print("categorical-like   :", categorical_like)
for c in categorical_like:
    train[c] = train[c].astype("category")
print("distinct values in occupancy:", n_unique["occupancy"], "| in co2_ppm:", n_unique["co2_ppm"])

## 2. What is missing — and *when*?

`isna()` tells us how much is missing. Plotting the missing share **over time** tells us *why*: a block of missing values
is a broken sensor; scattered ones are different.

In [ ]:
miss = train.isna().mean().round(4)
print(miss)

train["month"] = train["timestamp"].dt.month
fig, ax = plt.subplots(figsize=(8, 3))
train.groupby("month")["electricity_kwh"].apply(lambda s: s.isna().mean()).plot(kind="bar", ax=ax)
ax.set_ylabel("share missing"); ax.set_title("electricity_kwh: share missing per month")
plt.show()

### ✏️ Your turn 2

Which month is the worst for **CO₂** data?

Change the column in the `TODO` line from `"electricity_kwh"` to `"co2_ppm"` and run. The cell prints the missing share per
month, the worst month, and the longest *consecutive* run of missing values in that column.

**What to change** — edit the cell exactly like this, then run it:

Change `column = "electricity_kwh"` into `column = "co2_ppm"`

In [ ]:
column = "electricity_kwh"        # TODO: use "co2_ppm"

share_by_month = train.groupby("month")[column].apply(lambda s: s.isna().mean()).round(3)
print(share_by_month)
print("worst month:", share_by_month.idxmax(), "| share:", share_by_month.max())

is_na = train[column].isna().astype(int)
run_id = (is_na.diff() != 0).cumsum()
longest_run = is_na.groupby(run_id).sum().max()
print("longest run of consecutive missing hours:", longest_run)

## 3. Missing values in disguise

`describe()` shows suspicious extremes: a temperature of **9999 °C**, a negative number of people, hours with exactly
**zero** electricity consumption (a building never uses nothing).

In [ ]:
train[["electricity_kwh", "temperature_c", "occupancy", "co2_ppm"]].describe().round(1)

### ✏️ Your turn 3

Convert the disguised missing values to real `NaN`.

The dictionary `placeholders` lists, per column, the values that mean "no reading". At the moment only the temperature
placeholder is listed. Add:
- `occupancy`: the value `-1`
- `electricity_kwh`: the value `0` (exactly zero consumption is a logger fault)

Run the cell: it replaces them and reports how many were replaced and the new column summaries.

**What to change** — edit the cell exactly like this, then run it:

Replace
```python
    "occupancy": [],
    "electricity_kwh": [],
```
by
```python
    "occupancy": [-1],
    "electricity_kwh": [0],
```

In [ ]:
placeholders = {
    "temperature_c": [9999],
    "occupancy": [],              # TODO
    "electricity_kwh": [],        # TODO
}

clean = train.copy()
for col, values in placeholders.items():
    n_rep = clean[col].isin(values).sum()
    clean[col] = clean[col].replace(values, np.nan)
    print(f"{col:16s} placeholders replaced: {n_rep}")

print("mean temperature_c :", round(clean["temperature_c"].mean(), 2))
print("min occupancy      :", clean["occupancy"].min())
print("mean electricity   :", round(clean["electricity_kwh"].mean(), 2))
print("missing electricity share:", round(clean["electricity_kwh"].isna().mean(), 4))

## 4. When consumption is missing, it is cold

Is the missingness related to other variables? If yes, deleting the incomplete rows would distort the picture
(it removes the coldest and most interesting hours). We bin the temperature and compute the missing share of `electricity_kwh`
in each bin.

### ✏️ Your turn 4

The starter uses very coarse temperature bins. Replace the bin edges in the `TODO` line by

`[-30, -2, 2, 10, 40]`

(four bins: below −2 °C, −2 to 2 °C, 2 to 10 °C, above 10 °C) and run the cell.

**What to change** — edit the cell exactly like this, then run it:

Change `edges = [-30, 0, 20, 40]` into `edges = [-30, -2, 2, 10, 40]`

In [ ]:
edges = [-30, 0, 20, 40]          # TODO: [-30, -2, 2, 10, 40]

clean["temp_bin"] = pd.cut(clean["temperature_c"], edges)
by_bin = clean.groupby("temp_bin", observed=True)["electricity_kwh"].agg(
    hours="size", missing_share=lambda s: s.isna().mean())
print(by_bin.round(3))
print("missing share in the coldest bin:", round(by_bin["missing_share"].iloc[0], 3))
print("missing share in the warmest bin:", round(by_bin["missing_share"].iloc[-1], 3))

## 5. Distributions — and the danger of looking at too many variables

Skewness (`.skew()`) is 0 for a symmetric distribution, positive for a long right tail. Also check for **point masses**:
many observations at exactly one value (outside working hours the occupancy is near zero).

### ✏️ Your turn 5

Compute three numbers on the cleaned training data (`clean`):
- `skew_elec`: skewness of `electricity_kwh` (2 decimals)
- `skew_occ`: skewness of `occupancy` (2 decimals)
- `share_empty`: share of hours (ignoring missing) with `occupancy <= 5`, i.e. an essentially empty building (3 decimals)

**What to change** — edit the cell exactly like this, then run it:

Replace
```python
skew_elec = np.nan
skew_occ = np.nan
share_empty = np.nan
```
by
```python
skew_elec = round(clean["electricity_kwh"].skew(), 2)
skew_occ = round(clean["occupancy"].skew(), 2)
share_empty = round((clean["occupancy"].dropna() <= 5).mean(), 3)
```

In [ ]:
skew_elec = np.nan      # TODO
skew_occ = np.nan       # TODO
share_empty = np.nan    # TODO

print("skew electricity:", skew_elec)
print("skew occupancy  :", skew_occ)
print("share of hours with occupancy <= 5:", share_empty)

### Look at enough variables and some will look real

We add columns of pure **random noise** to the training set and count how many are "significantly" correlated with
electricity consumption (p < 0.05). By chance, about 5 % should be.

### ✏️ Your turn 6

Change `n_noise` from 20 to **200** and run again.

**What to change** — edit the cell exactly like this, then run it:

Change `n_noise = 20` into `n_noise = 200`

In [ ]:
from scipy import stats

n_noise = 20          # TODO: 200
rng = np.random.default_rng(0)
target = clean["electricity_kwh"].dropna()
noise = pd.DataFrame(rng.normal(size=(len(target), n_noise)), index=target.index)

pvals = np.array([stats.pearsonr(noise[c], target)[1] for c in noise.columns])
print("noise columns:", n_noise)
print("'significant' at p < 0.05:", int((pvals < 0.05).sum()), f"({(pvals < 0.05).mean():.1%})")
print("strongest |correlation| found:", round(max(abs(stats.pearsonr(noise[c], target)[0]) for c in noise.columns), 4))

## Summary

* `nunique` separates categories, constants and continuous variables; **codes stored as numbers are categories**.
* Missing values hide as `9999`, `-1`, `0` — find them with `describe()` and fix them *before* computing anything.
* Check **when** and **under which conditions** values are missing. Missing-when-cold is not random.
* With enough variables, some will look related by chance alone.